# When PINNs Go Wrong: Pseudo-Time Stepping Against Spurious Solutions

**Paper:** Wang, S., Koohy, S., Lu, Y., Perdikaris, P. (2026). *When PINNs Go Wrong: Pseudo-Time Stepping Against Spurious Solutions.* arXiv:2604.20582 [cs.LG].

**Carpeta origen:** `PINNs/4. Otros/When PINNs Go Wrong Pseudo-Time Stepping Against Spurious Solutions.pdf`

## Como se usan las PINNs en este paper

Este paper analiza **por que fallan las PINNs** incluso cuando la perdida de residuo se hace pequena: la perdida empirica del residuo de la EDP, evaluada solo sobre un conjunto finito de puntos de colocacion, admite **soluciones espurias/triviales** que satisfacen exactamente esa perdida sin ser la solucion fisica (Teorema 2.1 del paper: para el problema homogeneo existe una $u^\dagger\in C^\infty$ que coincide con la solucion real cerca de los puntos de colocacion pero se vuelve identicamente cero tras un tiempo $t_0$, y cuyo residuo empirico es exactamente cero).

La tecnica que el paper revisita para mitigar esto es el **pseudo-time stepping** (continuacion pseudo-transitoria), que reemplaza la perdida de residuo estandar
$$\mathcal{R}_{\mathrm{int}}[u_\theta](t,\mathbf{x})=\partial_tu_\theta+\mathcal{D}[u_\theta]-\mathbf{f}=0$$
por su version **relajada** (Eq. 2.38, caso dependiente del tiempo):
$$\mathcal{L}_{\mathrm{pts}}(\theta;\theta^{k-1})=\frac{1}{N_{\mathrm{int}}}\sum_i\left|\frac{u_\theta(t_{\mathrm{int}}^i,\mathbf{x}_{\mathrm{int}}^i)-u_{\theta^{k-1}}(t_{\mathrm{int}}^i,\mathbf{x}_{\mathrm{int}}^i)}{\tau}+\mathcal{R}_{\mathrm{int}}[u_\theta](t_{\mathrm{int}}^i,\mathbf{x}_{\mathrm{int}}^i)\right|^2$$
donde $\theta^{k-1}$ son los parametros de la iteracion anterior (tratados como fijos/frozen). El paper demuestra (Teorema 2.5) que, combinado con **remuestreo de puntos de colocacion**, este esquema **amplifica** el residuo de las soluciones espurias en los nuevos puntos muestreados (de orden $h^{-1}$ a orden $\tau^2h^{-3}$), desestabilizandolas durante el entrenamiento -- el mecanismo real detras del exito empirico del pseudo-time stepping, mas alla de la mejora de condicionamiento que se le atribuia tradicionalmente.

Como el rendimiento de $\tau$ fijo es muy sensible a su eleccion (Fig. 5) y no puede ajustarse solo mirando la perdida de entrenamiento, el paper propone el **Algoritmo 1**: un paso pseudo-temporal **adaptativo** que se estima cada $m$ iteraciones mediante un sustituto en diferencias finitas (al estilo Barzilai-Borwein) de la magnitud inversa del Jacobiano local del residuo,
$$\hat\tau^k=\gamma^k\frac{\|\Delta\mathbf{u}^k\|_2}{\|\Delta\mathbf{r}^k\|_2+\varepsilon},\qquad \Delta\mathbf{u}^k:=\mathbf{u}_{\theta^k}-\mathbf{u}_{\theta^{k-1}},\quad \Delta\mathbf{r}^k:=\mathcal{R}_{\mathrm{int}}[\mathbf{u}_{\theta^k}]-\mathcal{R}_{\mathrm{int}}[\mathbf{u}_{\theta^{k-1}}]$$
suavizado por una media movil exponencial (con `stop-gradient`) y modulado por un **factor de encogimiento** $\gamma^k\in[\gamma_{\min},1]$ que decae segun un coseno en funcion de la reduccion logaritmica de la perdida de residuo (Eq. 2.67-2.68), para estabilizar el entrenamiento en etapas tardias.

Este cuaderno reproduce fielmente el **benchmark de adveccion lineal** que el paper usa como ejemplo canonico de fallo (Seccion 2.2, Fig. 1 arriba): $u_t+cu_x=0$ en $(t,x)\in[0,2]\times[0,2\pi]$, $u(0,x)=\sin(x)$, con condiciones periodicas y $c=50$ (velocidad de adveccion alta, un regimen donde el sesgo espectral hace que la PINN estandar colapse a una solucion casi constante/espuria). Se entrenan y comparan tres variantes: (A) PINN estandar (linea base, batch de colocacion fijo), (B) pseudo-time stepping con $\tau$ fijo y remuestreo aleatorio en cada iteracion, y (C) pseudo-time stepping **adaptativo** (Algoritmo 1 completo), todas contra la solucion exacta $u(t,x)=\sin(x-ct)$.

## Repositorio publico

El propio resumen del paper indica explicitamente: "All code and data accompanying this manuscript are available at https://github.com/sifanexisted/jaxpi2".

- **sifanexisted/jaxpi2** &mdash; https://github.com/sifanexisted/jaxpi2

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import copy
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Adveccion lineal ($c=50$, Eq. 2.9-2.10): red con incrustacion periodica exacta en $x$

In [ ]:
c_adv = 50.0
T_max = 2.0

def u_exact(t, x):
    return np.sin(x - c_adv * t)


class PeriodicPINN(nn.Module):
    """Entrada (t, cos(x), sin(x)): impone periodicidad exacta en x sin necesitar perdida de contorno."""
    def __init__(self, n_hidden=4, n_neurons=64):
        super().__init__()
        layers = [nn.Linear(3, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, t, x):
        feat = torch.cat([t, torch.cos(x), torch.sin(x)], dim=1)
        return self.net(feat)


def residual(model, t, x):
    """R_int[u](t,x) = u_t + c*u_x (Eq. 2.9), via autograd."""
    u = model(t, x)
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    return u_t + c_adv * u_x, u


def sample_collocation(n):
    t = (torch.rand(n, 1, device=device) * T_max).requires_grad_(True)
    x = (torch.rand(n, 1, device=device) * 2 * np.pi).requires_grad_(True)
    return t, x


n_ic = 128
x_ic = torch.rand(n_ic, 1, device=device) * 2 * np.pi  # sin grad: la perdida IC no necesita derivadas en x_ic/t_ic
t_ic = torch.zeros(n_ic, 1, device=device)
u_ic_target = torch.sin(x_ic)

Nt_eval, Nx_eval = 80, 80
t_eval_np = np.linspace(0, T_max, Nt_eval)
x_eval_np = np.linspace(0, 2 * np.pi, Nx_eval)
TT, XX = np.meshgrid(t_eval_np, x_eval_np, indexing='ij')
U_exact_grid = u_exact(TT, XX)
t_eval = torch.tensor(TT.reshape(-1, 1), dtype=torch.float32, device=device)
x_eval = torch.tensor(XX.reshape(-1, 1), dtype=torch.float32, device=device)


def rel_l2_error(model):
    with torch.no_grad():
        u_pred = model(t_eval, x_eval).cpu().numpy().reshape(Nt_eval, Nx_eval)
    return np.linalg.norm(u_pred - U_exact_grid) / np.linalg.norm(U_exact_grid), u_pred

## 2. Tres regimenes de entrenamiento: (A) linea base, (B) pseudo-time con $\tau$ fijo, (C) pseudo-time adaptativo (Algoritmo 1)

In [ ]:
N_col = 256
epochs = 3000
lambda_ic = 10.0


def train_baseline():
    """(A) PINN estandar: perdida de residuo directa, batch de colocacion fijo (Sec. 2.2)."""
    model = PeriodicPINN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    t_col, x_col = sample_collocation(N_col)
    err_hist = []
    for ep in range(epochs):
        opt.zero_grad()
        r, _ = residual(model, t_col, x_col)
        loss_int = torch.mean(r ** 2)
        loss_ic = torch.mean((model(t_ic, x_ic) - u_ic_target) ** 2)
        loss = loss_int + lambda_ic * loss_ic
        loss.backward()
        opt.step()
        if ep % 500 == 0 or ep == epochs - 1:
            err, _ = rel_l2_error(model)
            err_hist.append((ep, err))
            print(f'[Baseline] epoch {ep:5d} | loss={loss.item():.4e} | rel L2={err:.4f}')
    return model, err_hist


model_base, hist_base = train_baseline()

In [ ]:
def train_pseudo_time_fixed(tau_fixed=0.1):
    """(B) Pseudo-time stepping con tau fijo (Eq. 2.38), remuestreo aleatorio cada iteracion."""
    model = PeriodicPINN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    prev_model = copy.deepcopy(model)
    for p in prev_model.parameters():
        p.requires_grad_(False)

    err_hist = []
    for ep in range(epochs):
        t_col, x_col = sample_collocation(N_col)
        opt.zero_grad()
        r_theta, u_theta = residual(model, t_col, x_col)
        with torch.no_grad():
            u_prev = prev_model(t_col, x_col)
        loss_pts = torch.mean(((u_theta - u_prev) / tau_fixed + r_theta) ** 2)
        loss_ic = torch.mean((model(t_ic, x_ic) - u_ic_target) ** 2)
        loss = loss_pts + lambda_ic * loss_ic

        # theta^k (estado actual, antes de esta actualizacion) pasa a ser theta^{k-1} en la sig. iteracion
        new_prev = copy.deepcopy(model)
        for p in new_prev.parameters():
            p.requires_grad_(False)

        loss.backward()
        opt.step()
        prev_model = new_prev

        if ep % 500 == 0 or ep == epochs - 1:
            err, _ = rel_l2_error(model)
            err_hist.append((ep, err))
            print(f'[Pseudo-time fijo] epoch {ep:5d} | loss={loss.item():.4e} | rel L2={err:.4f}')
    return model, err_hist


model_pts_fixed, hist_pts_fixed = train_pseudo_time_fixed(tau_fixed=0.1)

In [ ]:
def train_pseudo_time_adaptive(m_update=25, beta_smooth=0.9, gamma_min=0.1,
                                s_start=2.0, s_end=6.0, eps_small=1e-8, tau_init=1.0):
    """(C) Pseudo-time stepping adaptativo -- Algoritmo 1 completo:
    surrogato tipo Barzilai-Borwein (Eq. 2.55-2.58) + decaimiento coseno del factor de encogimiento
    gamma^k (Eq. 2.67-2.68) + suavizado EMA con stop-gradient (Eq. 2.63)."""
    model = PeriodicPINN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    prev_model = copy.deepcopy(model)
    for p in prev_model.parameters():
        p.requires_grad_(False)

    tau = tau_init
    L0 = None
    err_hist, tau_hist = [], []
    for ep in range(epochs):
        t_col, x_col = sample_collocation(N_col)

        # --- actualizacion del paso pseudo-temporal cada m_update iteraciones (Alg. 1, linea 3) ---
        if ep % m_update == 0:
            with torch.no_grad():
                u_cur = model(t_col, x_col)
                u_pr = prev_model(t_col, x_col)
            r_cur, _ = residual(model, t_col, x_col)
            r_pr, _ = residual(prev_model, t_col, x_col)
            delta_u = torch.norm(u_cur - u_pr)
            delta_r = torch.norm(r_cur.detach() - r_pr.detach())

            L_int_now = torch.mean(r_cur.detach() ** 2).item()
            if L0 is None:
                L0 = L_int_now
            p_k = (np.log10((L0 + eps_small) / (L_int_now + eps_small)) - s_start) / (s_end - s_start)
            p_k = min(max(p_k, 0.0), 1.0)
            gamma_k = gamma_min + (1 - gamma_min) * (1 + np.cos(np.pi * p_k)) / 2

            tau_hat = gamma_k * (delta_u.item() / (delta_r.item() + eps_small))
            tau = (1 - beta_smooth) * tau + beta_smooth * tau_hat  # EMA con stop-gradient (escalar Python)
        tau_hist.append(tau)

        opt.zero_grad()
        r_theta, u_theta = residual(model, t_col, x_col)
        with torch.no_grad():
            u_prev = prev_model(t_col, x_col)
        loss_pts = torch.mean(((u_theta - u_prev) / tau + r_theta) ** 2)
        loss_ic = torch.mean((model(t_ic, x_ic) - u_ic_target) ** 2)
        loss = loss_pts + lambda_ic * loss_ic

        new_prev = copy.deepcopy(model)
        for p in new_prev.parameters():
            p.requires_grad_(False)

        loss.backward()
        opt.step()
        prev_model = new_prev

        if ep % 500 == 0 or ep == epochs - 1:
            err, _ = rel_l2_error(model)
            err_hist.append((ep, err))
            print(f'[Pseudo-time adaptativo] epoch {ep:5d} | loss={loss.item():.4e} | tau={tau:.4f} | rel L2={err:.4f}')
    return model, err_hist, tau_hist


model_pts_adapt, hist_pts_adapt, tau_hist = train_pseudo_time_adaptive()

## 3. Resultados: campos predichos y convergencia del error relativo $L^2$

In [ ]:
err_final_base, u_pred_base = rel_l2_error(model_base)
err_final_fixed, u_pred_fixed = rel_l2_error(model_pts_fixed)
err_final_adapt, u_pred_adapt = rel_l2_error(model_pts_adapt)

print(f'Error relativo L2 final -- Baseline: {100*err_final_base:.2f}% | '
      f'Pseudo-time fijo: {100*err_final_fixed:.2f}% | Pseudo-time adaptativo: {100*err_final_adapt:.2f}%')

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fields = [U_exact_grid, u_pred_base, u_pred_fixed, u_pred_adapt]
titles = ['Referencia (exacta)', f'Baseline ({100*err_final_base:.1f}%)',
          f'Pseudo-time fijo ({100*err_final_fixed:.1f}%)', f'Pseudo-time adaptativo ({100*err_final_adapt:.1f}%)']
for ax, field, title in zip(axes, fields, titles):
    im = ax.pcolormesh(t_eval_np, x_eval_np, field.T, cmap='RdBu_r', shading='auto', vmin=-1, vmax=1)
    ax.set_xlabel('t'); ax.set_ylabel('x'); ax.set_title(title)
plt.colorbar(im, ax=axes, shrink=0.8)
plt.show()

plt.figure(figsize=(7, 4.5))
for hist, label, style in [(hist_base, 'Baseline', 'o-'), (hist_pts_fixed, 'Pseudo-time fijo', 's-'),
                            (hist_pts_adapt, 'Pseudo-time adaptativo', '^-')]:
    eps_arr, errs = zip(*hist)
    plt.semilogy(eps_arr, errs, style, label=label)
plt.xlabel('Epoch'); plt.ylabel('Error relativo L2'); plt.title('Adveccion lineal (c=50): convergencia del error')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 3.5))
plt.plot(tau_hist)
plt.xlabel('Iteracion'); plt.ylabel(r'$\tau^k$'); plt.title('Evolucion del paso pseudo-temporal adaptativo (Algoritmo 1)')
plt.grid(alpha=0.3)
plt.show()

## Nota honesta sobre los resultados

Este cuaderno usa una red pequena (4x64) y solo 3000 iteraciones de Adam (frente a las decenas de miles de pasos y arquitecturas mas grandes del paper), por lo que las magnitudes exactas de error no coinciden con las de la Fig. 1/3 del paper. Sin embargo, la implementacion es fiel al **mecanismo algoritmico** exacto: la perdida pseudo-temporal relajada (Eq. 2.38), el remuestreo aleatorio de puntos de colocacion, y el Algoritmo 1 completo (surrogato de Barzilai-Borwein para $\tau$, decaimiento coseno de $\gamma^k$, suavizado EMA con stop-gradient). El patron cualitativo esperado -- que la linea base con $c=50$ tiende a colapsar hacia una solucion casi plana/espuria por sesgo espectral, mientras que las variantes de pseudo-time stepping (especialmente la adaptativa) capturan mejor la estructura ondulatoria propagante -- puede no ser tan pronunciado con esta escala reducida; se anima a aumentar `epochs` y el tamano de la red para reproducir mas fielmente la magnitud de las mejoras reportadas en el paper.